# Контроль качества данных эксперимента 3

**Статус:** техническая инвентаризация и диагностический ноутбук
реокардиомонитора РНЦХ. Функции сердца здесь не рассчитываются.

Паспорт серии приведён в
[`10.10`](10.10_Паспорт_эксперимента_3.md). Дата остаётся открытым полем до
сверки с первичным протоколом. Постороннее исследование на другом приборе не
включается: ноутбук обрабатывает только записи, явно перечисленные во внешней
конфигурации.

Численные наблюдения ниже относятся к историческому запуску до миграции.
Выходы вычислительных ячеек очищены; повторный запуск на внешних данных
ещё не выполнен.
Ориентировочные дыхательные интервалы из старых графиков не используются как
количественная разметка.


In [ ]:
# Импорты и внешний контракт данных
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
CSV_ROOT = (DATA_ROOT / CONFIG["csv_subdir"]).resolve()
CSV_ROOT.relative_to(DATA_ROOT)

RECORDINGS = CONFIG["recordings"]
record_ids = [item["record_id"] for item in RECORDINGS]
relative_paths = [item["relative_path"] for item in RECORDINGS]
if not RECORDINGS or len(record_ids) != len(set(record_ids)):
    raise ValueError("recordings должен содержать уникальные непустые record_id")
if len(relative_paths) != len(set(relative_paths)):
    raise ValueError("Один relative_path нельзя назначать нескольким record_id")

SOURCE_COLUMNS = CONFIG["source_columns"]
CANONICAL_COLUMNS = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
if len(SOURCE_COLUMNS) != len(CANONICAL_COLUMNS):
    raise ValueError("source_columns должен описывать ровно восемь столбцов CSV")

ACTIVE_THRESHOLD_OHM = float(CONFIG["active_channel_threshold_ohm"])
if not np.isfinite(ACTIVE_THRESHOLD_OHM) or ACTIVE_THRESHOLD_OHM <= 0:
    raise ValueError("active_channel_threshold_ohm должен быть положительным")

plt.rcParams.update({
    "figure.figsize": (15, 9),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 13,
})


def resolve_record_path(relative_path):
    path = (DATA_ROOT / relative_path).resolve()
    path.relative_to(CSV_ROOT)
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла для записи: {relative_path}")
    return path


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_record(path):
    frame = pd.read_csv(path)
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError(
            "Схема CSV не совпадает с source_columns внешней конфигурации"
        )
    frame.columns = CANONICAL_COLUMNS
    frame = frame.apply(pd.to_numeric, errors="raise")
    values = frame.to_numpy(dtype=float)
    if len(frame) < 2 or not np.isfinite(values).all():
        raise ValueError("CSV пуст, слишком короток или содержит нечисловые значения")
    dt = np.diff(frame["time_s"].to_numpy(dtype=float))
    if not np.all(dt > 0):
        raise ValueError("TIME должен строго возрастать")
    return frame


## 1. Инвентаризация и соответствие файлов

В анализ входят только обезличенные `record_id`, явно перечисленные в
`KALMYKOV_EXP03_CONFIG`. Относительный путь обязан находиться внутри
разрешённого `csv_subdir`. Остальные CSV учитываются только числом при проверке
полноты конфигурации и не анализируются.

Полный SHA-256 определяет байт-в-байт совпадающие записи. Канонический файл
задаётся конфигурацией, а не выбирается автоматически по длине пути.


In [ ]:
# Инвентаризация разрешённой области и явно включённых записей
discovered = sorted(CSV_ROOT.rglob("*.csv"))
discovered_hashes = {path: file_sha256(path) for path in discovered}
copy_count_by_hash = pd.Series(list(discovered_hashes.values())).value_counts()

inventory_rows = []
listed_paths = set()
for spec in RECORDINGS:
    path = resolve_record_path(spec["relative_path"])
    listed_paths.add(path)
    data = read_record(path)
    dt = np.diff(data["time_s"].to_numpy(dtype=float))
    sha256 = discovered_hashes.get(path, file_sha256(path))
    inventory_rows.append({
        "record_id": spec["record_id"],
        "path": path,
        "sha256": sha256,
        "sha256_prefix": sha256[:16],
        "copies_in_allowed_root": int(copy_count_by_hash.get(sha256, 1)),
        "rows": len(data),
        "duration_s": float(data["time_s"].iloc[-1] - data["time_s"].iloc[0]),
        "fs_hz": float(1.0 / np.median(dt)),
        "role": spec.get("role", "не указана"),
        "protocol_reference": spec.get("protocol_reference", "не указана"),
        "expected_active_channels": spec.get("expected_active_channels", []),
    })

inventory = pd.DataFrame(inventory_rows)
inventory["duplicate_in_list"] = inventory.duplicated("sha256", keep=False)
if inventory["duplicate_in_list"].any():
    duplicate_ids = inventory.loc[inventory["duplicate_in_list"], "record_id"].tolist()
    raise ValueError(f"В recordings перечислены дублирующие записи: {duplicate_ids}")

unlisted_count = len(set(discovered) - listed_paths)
mapping_table = inventory[[
    "record_id", "role", "protocol_reference", "expected_active_channels",
    "duration_s", "fs_hz", "copies_in_allowed_root", "sha256_prefix",
]].copy()
mapping_table["duration_s"] = mapping_table["duration_s"].round(2)
mapping_table["fs_hz"] = mapping_table["fs_hz"].round(2)
mapping_table.columns = [
    "Запись", "Роль", "Ссылка на протокол", "Ожидаемые активные каналы",
    "Длительность, с", "Частота, Гц", "Копий в разрешённой области",
    "SHA-256, начало",
]
display(mapping_table.style.hide(axis="index").set_properties(**{"text-align": "left"}))
print(f"Явно включено записей: {len(inventory)}")
print(f"Других CSV в разрешённой области: {unlisted_count}")
print("Неуказанные CSV не анализируются.")


### Результат предыдущей файловой проверки

- запись `14-07-01` соответствует отдельному каналу 1, ТТРКГ;
- запись `14-09-42` соответствует отдельному каналу 2, боковой сборке;
- запись `14-17-16` содержит поочерёдное отключение каналов, тогда как в
  текстовом протоколе для этой пробы указано 14:08;
- запись `14-19-52` содержит совместный дыхательный протокол, тогда как в
  текстовом протоколе указано 14:17;
- четыре основные записи имели байт-в-байт копии в двух каталогах разрешённой
  области данных.

Установлено несоответствие двух временных обозначений в текстовом протоколе и
именах файлов. Его причина для дальнейшего анализа не требуется; соответствие
записей протокольным пробам должно быть окончательно подтверждено по первичному
журналу. Дата эксперимента по этим временам не определяется.


In [ ]:
record_by_id = dict(zip(inventory["record_id"], inventory["path"]))
spec_by_id = {item["record_id"]: item for item in RECORDINGS}


def path_for_role(role):
    matches = [item for item in RECORDINGS if item.get("role") == role]
    if len(matches) != 1:
        raise ValueError(f"Для роли {role!r} ожидается ровно одна запись")
    return record_by_id[matches[0]["record_id"]], matches[0]["record_id"]


def channel_stats(path, channels):
    data = read_record(path)
    result = []
    for channel in channels:
        base = data[f"base_{channel}_ohm"]
        qs = data[f"qs_{channel}_ohm"]
        rheo = data[f"rheo_{channel}_mohm"]
        result.append({
            "Канал": channel,
            "BASE median, Ом": base.median(),
            "BASE p01–p99, Ом": f"{base.quantile(0.01):.2f}–{base.quantile(0.99):.2f}",
            "QS median, Ом": qs.median(),
            "QS max, Ом": qs.max(),
            "RHEO p01–p99, мОм": f"{rheo.quantile(0.01):.1f}–{rheo.quantile(0.99):.1f}",
        })
    table = pd.DataFrame(result)
    for column in ["BASE median, Ом", "QS median, Ом", "QS max, Ом"]:
        table[column] = table[column].round(2)
    return table


def plot_record(path, title, channels):
    data = read_record(path)
    time = data["time_s"]
    fs = 1.0 / np.median(np.diff(time))
    window = max(1, int(round(fs)))
    if window % 2 == 0:
        window += 1

    fig, axes = plt.subplots(
        4, 1, figsize=(15, 10), sharex=True,
        gridspec_kw={"height_ratios": [2.2, 1, 1, 1]},
    )
    colors = {1: "#2563eb", 2: "#dc2626"}
    for channel in channels:
        rheo = data[f"rheo_{channel}_mohm"]
        smooth = rheo.rolling(window, center=True, min_periods=1).median()
        axes[0].plot(time, rheo, color=colors[channel], alpha=0.16, linewidth=0.5)
        axes[0].plot(
            time, smooth, color=colors[channel], linewidth=1.7,
            label=f"канал {channel}, медиана 1 с",
        )
        axes[1].plot(
            time, data[f"base_{channel}_ohm"], color=colors[channel],
            linewidth=1.2, label=f"BASE {channel}",
        )
        axes[2].plot(
            time, data[f"qs_{channel}_ohm"], color=colors[channel],
            linewidth=1.2, label=f"QS {channel}",
        )

    axes[3].plot(time, data["ecg_v"], color="#111827", linewidth=0.7, label="ЭКГ")
    axes[0].set_ylabel("RHEO, мОм")
    axes[1].set_ylabel("BASE, Ом")
    axes[2].set_ylabel("QS, Ом")
    axes[3].set_ylabel("ЭКГ, В")
    axes[3].set_xlabel("Время от начала CSV, с")
    axes[0].set_title(title)
    for axis in axes:
        axis.legend(loc="upper right", ncol=max(1, len(channels)))
        axis.margins(x=0)
    fig.tight_layout()
    plt.show()


def active_intervals(path, threshold_ohm=ACTIVE_THRESHOLD_OHM):
    data = read_record(path)
    time = data["time_s"].to_numpy()
    base_1 = data["base_1_ohm"].to_numpy()
    base_2 = data["base_2_ohm"].to_numpy()
    state = (base_1 > threshold_ohm).astype(int) + 2 * (base_2 > threshold_ohm).astype(int)
    cuts = np.r_[0, np.flatnonzero(state[1:] != state[:-1]) + 1, len(state)]
    names = {0: "ни один", 1: "только 1", 2: "только 2", 3: "оба"}
    rows = []
    for left, right in zip(cuts[:-1], cuts[1:]):
        if time[right - 1] - time[left] < 0.25:
            continue
        rows.append({
            "Начало, с": time[left],
            "Конец, с": time[right - 1],
            "Активны по порогу": names[int(state[left])],
            "BASE1 median, Ом": np.median(base_1[left:right]),
            "BASE2 median, Ом": np.median(base_2[left:right]),
            "QS1 median, Ом": np.median(data["qs_1_ohm"].iloc[left:right]),
            "QS2 median, Ом": np.median(data["qs_2_ohm"].iloc[left:right]),
        })
    return pd.DataFrame(rows).round(2)


## 2. Отдельная запись ТТРКГ, канал 1

По протоколу запись содержит свободное дыхание, задержку после вдоха,
форсированное дыхание и задержку после выдоха. Старые численные границы этих
этапов служили только визуальными ориентирами. Канонические интервалы должна
создавать серия `11.11` с ручным контролем.


In [ ]:
path, record_id = path_for_role("ttrkg_channel_1_only")
display(channel_stats(path, [1]).style.hide(axis="index"))
plot_record(path, f"{record_id}: ТТРКГ, канал 1 отдельно", [1])


**Наблюдение предыдущего запуска.** Медиана `BASE1` составляла около
101 Ом, тогда как в примечании протокола указано около 80 Ом. `QS1` сохранял
значение 4700 Ом на протяжении записи. Такой же код встречался у отключённых
каналов, но его физический смысл и статус предела шкалы не установлены.
Поэтому запись требует аппаратной расшифровки `QS`; причина расхождения `BASE1`
не определяется этим наблюдением.


## 3. Отдельная запись боковой сборки, канал 2

По протоколу запись содержит свободное дыхание, задержку после вдоха, три
форсированных цикла и задержку после выдоха. Старые границы не передаются в
количественный анализ без разметки `11.11`.


In [ ]:
path, record_id = path_for_role("side_channel_2_only")
display(channel_stats(path, [2]).style.hide(axis="index"))
plot_record(path, f"{record_id}: боковая сборка, канал 2 отдельно", [2])


**Наблюдение предыдущего запуска.** Медиана `BASE2` составляла около
37,2 Ом и была близка к значению 36 Ом из протокола. `QS2` находился около
343 Ом и не принимал значение 4700 Ом. `BASE1` был равен нулю, что согласуется
с указанным отключением канала 1. Близость одного базового уровня протоколу не
заменяет калибровку прибора.


## 4. Запись поочерёдного отключения каналов

Кандидатные состояния определяются по порогу `BASE`, заданному во внешней
конфигурации. Порог является техническим правилом сегментации, а не
физическим критерием исправности. Переходы и подписи состояний должны быть
проверены по протоколу до использования численных различий.


In [ ]:
path, record_id = path_for_role("channel_switch_test")
switch_intervals = active_intervals(path)
display(switch_intervals.style.hide(axis="index"))
plot_record(path, f"{record_id}: поочерёдное отключение каналов", [1, 2])


**Наблюдение предыдущего запуска.** При совместной работе медианы
составляли приблизительно `BASE1=54 Ом` и `BASE2=41 Ом`. После отключения
канала 1 значение `BASE2` оставалось около 37 Ом; после отключения канала 2
`BASE1` возрастало примерно до 93 Ом.

Таким образом, в этой записи установлен факт зависимости показаний
оставшегося канала от состояния другого канала. Причина эффекта не определена;
шунтирование, перераспределение тока, коммутация и алгоритм прибора остаются
гипотезами. Значение `QS=4700` встречалось у отключённого канала и у канала 1
при одиночной работе, однако его физическая интерпретация пока неизвестна.


## 5. Совместная запись обоих каналов с дыхательным протоколом

В записи визуально присутствуют свободное дыхание, задержка после вдоха,
форсированные циклы и задержка после выдоха. Точные интервалы не задаются в
этом ноутбуке контроля качества и должны поступать из принятой разметки `11.11`.


In [ ]:
path, record_id = path_for_role("both_channels_breathing")
display(channel_stats(path, [1, 2]).style.hide(axis="index"))
plot_record(path, f"{record_id}: оба канала, дыхательный протокол", [1, 2])


**Наблюдение предыдущего запуска.** При совместной работе медианы
составляли приблизительно `BASE1=54,4 Ом` и `BASE2=40,7 Ом`; второй уровень
был близок к протокольному, первый — ниже указанного значения 60 Ом.
`QS1` находился около 1307 Ом, а `QS2` — около 368 Ом.

Изменения обоих `RHEO` визуально совпадали по времени с дыхательными
манёврами. Это не устанавливает анатомический источник сигнала. На крупных
дыхательных изменениях наблюдались участки у границы записанного диапазона;
наличие и механизм насыщения требуют аппаратной проверки.


## 6. Выводы и границы результата

### Подтверждённые файловые и экспериментальные факты

1. В разрешённой области данных обнаружены байт-в-байт копии четырёх основных
   записей; расчёт должен использовать каждый полный SHA-256 один раз.
2. В записи поочерёдного отключения показания оставшегося канала зависят от
   состояния другого канала. Механизм эффекта не установлен.
3. Временные обозначения двух записей не совпадают с текстовым протоколом.
4. Постороннее несинхронное исследование на другом приборе не относится к
   эксперименту 3.

### Диагностические наблюдения предыдущего запуска

- отдельная запись канала 2 имела `BASE2`, близкий к протокольному значению, и
  стабильный `QS2` без значения 4700 Ом;
- отдельная запись канала 1 имела `BASE1` выше протокольного значения и
  постоянный код `QS1=4700 Ом`;
- физический смысл значения 4700 Ом и возможное насыщение `RHEO` не
  установлены.

### Требования к продолжению

1. Заполнить явный список допустимых записей и точные исходные названия
   столбцов во внешней конфигурации, затем повторно выполнить ноутбук.
2. Сверить дату и соответствие протокольных проб по первичному журналу.
3. Передать дыхательные интервалы и ЭКГ-события сериям `11.11` и `11.12`, не
   используя старые ориентировочные границы как принятую разметку.
4. Проверить `QS`, пределы диапазона и межканальное влияние по отдельному
   аппаратному протоколу. Числа версии РНЦХ не использовать как поправку для
   версии МГТУ.


In [ ]:
# @title Файловый QC и кандидатный манифест
from record_qc import build_exp03_candidate_manifest

QC_MANIFEST_PATH, QC_MANIFEST = build_exp03_candidate_manifest(CONFIG_PATH)
QC_INCLUDED = [item for item in QC_MANIFEST["records"] if item["include"]]
QC_UNCLASSIFIED = [
    item for item in QC_MANIFEST["records"]
    if item["qc_status"] == "unclassified_not_included_pending_primary_protocol"
]
print("10.11 file_qc_status: pending_manual_review")
print("Основных включённых записей:", len(QC_INCLUDED))
print("Неклассифицированных уникальных записей:", len(QC_UNCLASSIFIED))
print("Кандидатный манифест:", QC_MANIFEST_PATH)


## Производный манифест записей

Последующие расчёты серии 40 принимают только обезличенный QC-манифест по схеме
`schemas/record_manifest.schema.json`. Он должен фиксировать точный `record_id`,
SHA-256 исходной записи, добровольца, конфигурацию, монтаж, размер боковой сборки
и **фактически**, а не ожидаемо активные каналы. До ручного принятия такого
манифеста реальный расчёт серии 40 блокируется.
